# 🏋️ FitMind — QLoRA Fine-tuning (Qwen3-1.7B)

Trains a calorie-tracking SLM on the FitMind dataset using QLoRA.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `train_chat.jsonl` and `val_chat.jsonl` (Cell 3 has an uploader)

Run cells top to bottom. Total time ~2-3 hours.

## Cell 1 — Install dependencies

In [ ]:
!pip install -q -U "transformers>=4.51" "trl>=0.12" "peft>=0.13" \
    "datasets>=3.0" "accelerate>=1.0" "bitsandbytes>=0.44"
print('Dependencies installed. Restart NOT required.')

## Cell 2 — Check GPU
Should show a Tesla T4 with ~15GB.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch to T4!')
print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1) if torch.cuda.is_available() else 0)

## Cell 3 — Upload your data
Click and select **train_chat.jsonl** and **val_chat.jsonl** from your laptop.

In [ ]:
from google.colab import files
import os
uploaded = files.upload()  # pick train_chat.jsonl and val_chat.jsonl
print('\nUploaded:', list(uploaded.keys()))
assert os.path.exists('train_chat.jsonl'), 'train_chat.jsonl missing!'
assert os.path.exists('val_chat.jsonl'), 'val_chat.jsonl missing!'
print('Both files present.')

## Cell 4 — Load dataset

In [ ]:
from datasets import load_dataset

data = load_dataset('json', data_files={
    'train': 'train_chat.jsonl',
    'val':   'val_chat.jsonl'
})
print(data)
print('\nFirst training conversation:')
for m in data['train'][0]['messages']:
    print(f"[{m['role']}] {m['content'][:120]}...")

## Cell 5 — Load Qwen3-1.7B in 4-bit (the 'Q' in QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen3-1.7B'

# 4-bit quantization config (NF4) — lets a 1.7B model train on a free T4
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False  # needed for training
print('Model loaded in 4-bit.')

## Cell 6 — LoRA config (the small trainable adapter)
`r=16` = rank. Only these tiny matrices train; the base stays frozen.

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)
print('LoRA config ready (r=16, alpha=32).')

## Cell 7 — Training config + trainer
SFTTrainer applies Qwen's chat template to the `messages` automatically.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir='fitmind-lora',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy='epoch',
    save_strategy='epoch',
    bf16=True,
    max_length=1024,
    packing=False,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=data['train'],
    eval_dataset=data['val'],
    peft_config=lora_config,
    processing_class=tokenizer,
)
print('Trainer ready. Trainable params:')
trainer.model.print_trainable_parameters()

## Cell 8 — TRAIN 🚀 (~2-3 hours)
Watch `loss` go down. If it drops from ~2.0 to under ~0.5, it's learning well.

In [ ]:
trainer.train()
print('\nTraining complete!')

## Cell 9 — Save the LoRA adapter

In [ ]:
trainer.save_model('fitmind-lora-final')
tokenizer.save_pretrained('fitmind-lora-final')
print('Adapter saved to fitmind-lora-final/')
!du -sh fitmind-lora-final

## Cell 10 — Quick test (does it work?)
Try a fresh food sentence the model never saw.

In [ ]:
from transformers import pipeline

SYSTEM = ("You are FitMind, an offline calorie tracker for Indian gym users. "
          "The user logs food in English, Hindi, or Hinglish. Extract the food "
          "items, estimate calories as a [min, max] range, sum the macros, add "
          "this meal to the running daily total from the context, and compute "
          "the remaining budget. Reply briefly in the SAME language as the user. "
          "Always answer with a single JSON object.")

test_user = ("Daily budget: 2000 cal\nEaten today: nothing yet\nTotal so far: 0 cal\n\n"
             "User (hinglish): abhi maine 3 roti aur ek bowl rajma khaya")

messages = [{'role':'system','content':SYSTEM},{'role':'user','content':test_user}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)

import torch
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))

## Cell 11 — Download adapter to your laptop
Zips and downloads the trained adapter (~30-60 MB).

In [ ]:
!zip -r fitmind-lora-final.zip fitmind-lora-final
from google.colab import files
files.download('fitmind-lora-final.zip')

## Cell 12 — Full evaluation on test set
Upload **test_chat.jsonl** first (Files panel or the uploader below).
Runs the model on all 38 unseen examples and scores it.

In [ ]:
import json, os, re, torch

# Upload test_chat.jsonl if not already present
if not os.path.exists('test_chat.jsonl'):
    from google.colab import files
    files.upload()  # pick test_chat.jsonl

test = [json.loads(l) for l in open('test_chat.jsonl', encoding='utf-8')]
print(f'Evaluating on {len(test)} unseen examples...\n')

def detect_lang(text):
    if re.search(r'[ऀ-ॿ]', text): return 'hindi'
    hinglish = {'khaya','maine','abhi','bacha','aur','kha','liye','raha','subah','khaayi','khaye'}
    if len(set(text.lower().split()) & hinglish) >= 1: return 'hinglish'
    return 'english'

n = len(test)
json_valid = item_match = lang_match = 0
cal_errors = []

for i, ex in enumerate(test):
    msgs = ex['messages']
    system, user = msgs[0]['content'], msgs[1]['content']
    expected = json.loads(msgs[2]['content'])

    prompt = tokenizer.apply_chat_template(
        [{'role':'system','content':system},{'role':'user','content':user}],
        tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id)
    pred_text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:],
                                 skip_special_tokens=True).strip()

    # 1. JSON valid?
    try:
        s, e = pred_text.find('{'), pred_text.rfind('}')+1
        pred = json.loads(pred_text[s:e])
        json_valid += 1
    except Exception:
        print(f'[{i}] BAD JSON: {pred_text[:80]}')
        continue

    # 2. Item match (food names)
    exp_items = set(it.get('name','') for it in expected.get('items',[]))
    pred_items = set(it.get('name','') for it in pred.get('items',[]))
    if exp_items == pred_items:
        item_match += 1

    # 3. Calorie error (midpoint difference)
    if isinstance(pred.get('calories'), list) and len(pred['calories'])==2:
        exp_mid = sum(expected['calories'])/2
        pred_mid = sum(pred['calories'])/2
        cal_errors.append(abs(exp_mid - pred_mid))

    # 4. Language match (reply language vs expected input language)
    exp_lang = detect_lang(user.split('User (')[-1])
    pred_lang = detect_lang(pred.get('reply',''))
    if exp_lang == pred_lang:
        lang_match += 1

import statistics
print('\n' + '='*45)
print('EVALUATION RESULTS (38 unseen test examples)')
print('='*45)
print(f'JSON valid:        {json_valid}/{n}  ({100*json_valid/n:.0f}%)')
print(f'Item extraction:   {item_match}/{n}  ({100*item_match/n:.0f}%)')
print(f'Language match:    {lang_match}/{n}  ({100*lang_match/n:.0f}%)')
if cal_errors:
    print(f'Calorie error MAE: {statistics.mean(cal_errors):.0f} cal (avg off)')
    print(f'Calorie error med: {statistics.median(cal_errors):.0f} cal')